In [ ]:
#| hide
#| default_exp nbio

# nbio

> Reading and writing Jupyter notebooks

In [ ]:
#| export
from fastcore.basics import *
from fastcore.xtras import rtoken_hex
from fastcore.imports import *
from fastcore.ansi import ansi2html

import ast,functools
from pprint import pformat,pprint
from json import loads,dumps

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq

## Reading a notebook

A notebook is just a json file.

In [ ]:
#| exports
def _read_json(self, encoding=None, errors=None):
    return loads(Path(self).read_text(encoding=encoding, errors=errors))

In [ ]:
minimal_fn = Path('../tests/minimal.ipynb')
minimal_txt = AttrDict(_read_json(minimal_fn))

It contains two sections, the `metadata`...:

In [ ]:
minimal_txt.metadata

{'solveit_dialog_mode': 'learning', 'solveit_ver': 2}

...and, more importantly, the `cells`:

In [ ]:
minimal_txt.cells

[{'cell_type': 'markdown',
  'id': '801558df',
  'metadata': {},
  'source': ['## A minimal notebook']},
 {'cell_type': 'code',
  'execution_count': None,
  'id': 'e2147a69',
  'metadata': {'time_run': '2026-01-04T20:52:49.901559+00:00'},
  'outputs': [{'data': {'text/plain': ['2']},
    'execution_count': 0,
    'metadata': {},
    'output_type': 'execute_result'}],
  'source': ['# Do some arithmetic\n', '1+1']}]

The second cell here is a `code` cell, however it contains no outputs, because it hasn't been executed yet. To execute a notebook, we first need to convert it into a format suitable for `nbclient` (which expects some `dict` keys to be available as attrs, and some available as regular `dict` keys). Normally, `nbformat` is used for this step, but it's rather slow and inflexible, so we'll write our own function based on `fastcore`'s handy `dict2obj`, which makes all keys available as both attrs *and* keys.

In [ ]:
#| export
class NbCell(AttrDict):
    def __init__(self, idx, cell):
        super().__init__(cell)
        self.idx_ = idx
        if 'id' not in self: self.id = rtoken_hex(4)
        if 'source' in self: self.set_source(self.source)

    def set_source(self, source):
        self.source = ''.join(source)
        if '_parsed_' in self: del(self['_parsed_'])

    def parsed_(self):
        if self.cell_type!='code' or self.source.strip()[:1] in ['%', '!']: return
        if '_parsed_' not in self:
            try: self._parsed_ = ast.parse(self.source).body
            # you can assign the result of ! to a variable in a notebook cell
            # which will result in a syntax error if parsed with the ast module.
            except SyntaxError: return
        return self._parsed_

    def __hash__(self): return hash(self.source) + hash(self.cell_type)
    def __eq__(self,o): return self.source==o.source and self.cell_type==o.cell_type

We use an `AttrDict` subclass which has some basic functionality for accessing notebook cells.

In [ ]:
#| export
def _dict2obj(d, list_func=list, dict_func=AttrDict):
    "Convert (possibly nested) dicts (or lists of dicts) to `AttrDict`"
    if isinstance(d, list): return list(map(_dict2obj, d))
    if not isinstance(d, dict): return d
    return dict_func(**{k:_dict2obj(v) for k,v in d.items()})

def dict2nb(js=None, **kwargs):
    "Convert dict `js` to an `AttrDict`, "
    nb = _dict2obj(js or kwargs)
    nb.cells = [NbCell(*o) for o in enumerate(nb.cells)]
    return nb

We can now convert our JSON into this `nbclient`-compatible format, which pretty prints the source code of cells in notebooks.

In [ ]:
minimal = dict2nb(minimal_txt)
cell = minimal.cells[1]
cell

```python
{ 'cell_type': 'code',
  'execution_count': None,
  'id': 'e2147a69',
  'idx_': 1,
  'metadata': {'time_run': '2026-01-04T20:52:49.901559+00:00'},
  'outputs': [ { 'data': {'text/plain': ['2']},
                 'execution_count': 0,
                 'metadata': {},
                 'output_type': 'execute_result'}],
  'source': '# Do some arithmetic\n1+1'}
```

The abstract syntax tree of source code cells is available in the `parsed_` property:

In [ ]:
cell.parsed_(), cell.parsed_()[0].value.op

([<ast.Expr>], <ast.Add>)

In [ ]:
#| export
def read_nb(path):
    "Return notebook at `path`"
    res = dict2nb(_read_json(path, encoding='utf-8'))
    res['path_'] = str(path)
    return res

This reads the JSON for the file at `path` and converts it with `dict2nb`. For instance:

In [ ]:
minimal = read_nb(minimal_fn)
str(minimal.cells[0])

"{'cell_type': 'markdown', 'id': '801558df', 'metadata': {}, 'source': '## A minimal notebook', 'idx_': 0}"

The file name read is stored in `path_`:

In [ ]:
minimal.path_

'../tests/minimal.ipynb'

## Creating a notebook

In [ ]:
#| export
def mk_cell(text,  # `source` attr in cell
            cell_type='code',  # `cell_type` attr in cell
            **kwargs):  # any other attrs to add to cell
    "Create an `NbCell` containing `text`"
    assert cell_type in {'code', 'markdown', 'raw'}
    if 'metadata' not in kwargs: kwargs['metadata']={}
    if cell_type == 'code':
        kwargs['outputs']=[]
        kwargs['execution_count']=0
    return NbCell(0, dict(cell_type=cell_type, source=text, directives_={}, **kwargs))

In [ ]:
mk_cell('print(1)', execution_count=0)

```python
{ 'cell_type': 'code',
  'directives_': {},
  'execution_count': 0,
  'id': '2e6586af',
  'idx_': 0,
  'metadata': {},
  'outputs': [],
  'source': 'print(1)'}
```

In [ ]:
#| export
def new_nb(cells=None, meta=None, nbformat=4, nbformat_minor=5):
    "Returns an empty new notebook"
    cells = [o if isinstance(o,dict) else mk_cell(o) for o in cells]
    return dict2nb(cells=cells or [],metadata=meta or {},nbformat=nbformat,nbformat_minor=nbformat_minor)

Use this function when creating a new notebook. Useful for when you don't want to create a notebook on disk first and then read it.

## Writing a notebook

In [ ]:
#| export
def nb2dict(d, k=None):
    "Convert parsed notebook to `dict`"
    if k=='source': return d.splitlines(keepends=True)
    if isinstance(d, list): return list(map(nb2dict,d))
    if not isinstance(d, dict): return d
    return dict(**{k:nb2dict(v,k) for k,v in d.items() if k[-1] != '_'})

This returns the exact same dict as is read from the notebook JSON.

In [ ]:
minimal_fn = Path('../tests/minimal.ipynb')
minimal = read_nb(minimal_fn)
minimal_dict = _read_json(minimal_fn)
assert minimal_dict==nb2dict(minimal)

In [ ]:
#| export
def nb2str(nb):
    "Convert `nb` to a `str`"
    if isinstance(nb, (AttrDict,list)): nb = nb2dict(nb)
    return dumps(nb, sort_keys=True, indent=1, ensure_ascii=False) + "\n"

To save a notebook we first need to convert it to a `str`:

In [ ]:
print(nb2str(minimal)[:45])

{
 "cells": [
  {
   "cell_type": "markdown",


In [ ]:
#| export
def write_nb(nb, path):
    "Write `nb` to `path`"
    new = nb2str(nb)
    path = Path(path)
    old = Path(path).read_text(encoding='utf-8') if path.exists() else None
    if new!=old:
        with open(path, 'w', encoding='utf-8') as f: f.write(new)

This returns the exact same string as saved by Jupyter.

In [ ]:
tmp = Path('tmp.ipynb')
try:
    minimal_txt = minimal_fn.read_text()
    write_nb(minimal, tmp)
    test_eq(minimal_txt, tmp.read_text())
finally: tmp.unlink()

Here's how to put all the pieces of `fastcore.nbio` together:

In [ ]:
nb = new_nb([mk_cell('print(1)')])
path = Path('test.ipynb')
write_nb(nb, path)
nb2 = read_nb(path)
print(nb2.cells)
path.unlink()

[{'cell_type': 'code', 'execution_count': 0, 'id': '4d3f373b', 'metadata': {}, 'outputs': [], 'source': 'print(1)', 'idx_': 0}]


## Notebook class

In [ ]:
#| export
from fastcore.xml import Code,Markdown,Raw,NB,Source,Out,to_xml

In [ ]:
#| export
_ctfuns_nb = dict(code=Code, markdown=Markdown, raw=Raw)

def cell2xml(cell, ids=True, incl_out=True):
    "Convert NbCell to concise XML format"
    f = _ctfuns_nb[cell.cell_type]
    kw = dict(id=cell.id) if ids else {}
    output = getattr(cell, 'outputs', None) if incl_out else None
    if not output: return f(cell.source, **kw)
    return f(Source(cell.source), Out(str(output)), **kw)

def cells2xml(cells, wrap=NB, ids=True, incl_out=True, **kw):
    "Convert notebook cells to XML format"
    return to_xml(wrap(*[cell2xml(c, ids=ids, incl_out=incl_out) for c in cells], **kw), do_escape=False)


We can view any notebook as concise XML. For instance, our minimal notebook:

In [ ]:
print(cells2xml(nb.cells))

<nb><code id="4d3f373b">print(1)</code></nb>


In [ ]:
repr(cell2xml(nb.cells[0]))

'<code id="4d3f373b">print(1)</code>'

In [ ]:
#| export
class Notebook:
    "Read, query, and edit Jupyter notebooks"
    def __init__(self, nb, path=None):
        if not path: path = Path("unnamed.ipynb")
        store_attr()

    @property
    def _id2cell(self): return {c.id: c for c in self.nb.cells}

    @classmethod
    def open(cls, path):
        path = Path(path).resolve()
        return cls(read_nb(path), path)

    def save(self, path=None): write_nb(self.nb, path or self.path)

    @property
    def cells(self): return self.nb.cells
    @property
    def meta(self): return self.nb.metadata

    def __getitem__(self, k): return self.cells[k] if isinstance(k, (int, slice)) else self._id2cell[k]
    def __setitem__(self, k, source): self[k].set_source(source)
    def __len__(self): return len(self.cells)
    def __iter__(self): return iter(self.cells)
    def __contains__(self, k): return k in self._id2cell
    def __delitem__(self, k): self.cells.remove(self[k])
    def __repr__(self): return cells2xml(self.cells, path=self.path)
    @property
    def concise (self): return cells2xml(self.cells, path=self.path.name, incl_out=False)

We can now open a notebook and access its metadata and cells:

In [ ]:
nbo = Notebook.open(minimal_fn)
list(nbo.meta), len(nbo.cells), len(nbo)

(['solveit_dialog_mode', 'solveit_ver'], 2, 2)

In [ ]:
nbo.path.name

'minimal.ipynb'

In [ ]:
[o.id for o in nbo]

['801558df', 'e2147a69']

In [ ]:
'e2147a69' in nbo, 'nonexistent' in nbo

(True, False)

Notebooks' repr is their xml:

In [ ]:
nbo

<nb path="/Users/jhoward/aai-ws/fastcore/tests/minimal.ipynb"><markdown id="801558df">## A minimal notebook</markdown><code id="e2147a69"><source># Do some arithmetic
1+1<out>[{'data': {'text/plain': ['2']}, 'execution_count': 0, 'metadata': {}, 'output_type': 'execute_result'}]</out></code></nb>

You can also get a more concise version that doesn't include outputs or the full path:

In [ ]:
print(nbo.concise)

<nb path="minimal.ipynb"><markdown id="801558df">## A minimal notebook</markdown><code id="e2147a69"># Do some arithmetic
1+1</code></nb>


Cells can be accessed by integer index or by their string id:

In [ ]:
nbo[0].source

'## A minimal notebook'

In [ ]:
nbo['e2147a69'].source

'# Do some arithmetic\n1+1'

You can directly set a cell's source by id or index:

In [ ]:
nbo['e2147a69'] = '2+2'
nbo['e2147a69'].source


'2+2'

You can also update outputs and metadata directly on a cell:

In [ ]:
nbo['e2147a69'].outputs = [{'output_type': 'execute_result', 'data': {'text/plain': ['4']}}]
nbo['e2147a69'].outputs

[{'output_type': 'execute_result', 'data': {'text/plain': ['4']}}]

In [ ]:
nbo['e2147a69'].metadata['custom'] = True
nbo['e2147a69'].metadata

```python
{'custom': True, 'time_run': '2026-01-04T20:52:49.901559+00:00'}
```

The `add` method inserts a new cell at a given position (defaulting to the end):

In [ ]:
#| export
@patch
def add(self:Notebook, source, cell_type='code', idx=None, after=None, before=None, **kwargs):
    "Add a new cell with `source` at `idx` (default: end), or `after`/`before` a cell id"
    if after: idx = next((i for i,c in enumerate(self.cells) if c.id==after), None)
    if idx is None and after: raise KeyError(after)
    if idx is not None: idx += 1
    elif before: idx = next((i for i,c in enumerate(self.cells) if c.id==before), None)
    if idx is None and before: raise KeyError(before)
    c = mk_cell(source, cell_type=cell_type, **kwargs)
    if idx is None: self.cells.append(c)
    else: self.cells.insert(idx, c)
    return c

In [ ]:
nbo.add('print("hello")')
nbo.add('# A heading', cell_type='markdown', idx=0)
len(nbo), nbo[0].source

(4, '## A minimal notebook')

Cells can also be inserted relative to an existing cell by id:

In [ ]:
cid = nbo[0].id
nbo.add('# After first', cell_type='markdown', after=cid)
nbo.add('# Before first', cell_type='markdown', before=cid)
[c.source for c in nbo[:3]]

['# Before first', '## A minimal notebook', '# After first']

In [ ]:
#| export
@patch
def md(self:Notebook, source, idx=None, after=None, before=None, **kwargs):
    "Add a new cell with `source` at `idx` (default: end), or `after`/`before` a cell id"
    return self.add(source, cell_type='markdown', idx=idx, after=after, before=before, **kwargs)

`md` is a shortcut to `add(..., cell_type='markdown')`

In [ ]:
nbo.md('A note')
len(nbo), nbo[-1].cell_type

(7, 'markdown')

You can delete by id or index:

In [ ]:
prev_len = len(nbo)
del nbo[0]
len(nbo) == prev_len - 1


True

The `find` method searches cell sources by regex, returning matching cells:

In [ ]:
#| export
@patch
def find(self:Notebook, pat, cell_type=None):
    "Find cells with source matching regex `pat`"
    return [c for c in self.cells if re.search(pat, c.source) and (not cell_type or c.cell_type==cell_type)]

In [ ]:
nbo.find(r'\d\+\d', cell_type='code')

[{'cell_type': 'code',
  'execution_count': None,
  'id': 'e2147a69',
  'metadata': {'time_run': '2026-01-04T20:52:49.901559+00:00', 'custom': True},
  'outputs': [{'output_type': 'execute_result',
    'data': {'text/plain': ['4']}}],
  'source': '2+2',
  'idx_': 1}]

In [ ]:
#| export
@patch
def move(self:Notebook, src_ids, after=None, before=None):
    "Move cells with `src_ids` after/before a cell id, or to end"
    cells = [self[k] for k in listify(src_ids)]
    for c in cells: self.cells.remove(c)
    if after: idx = next((i+1 for i,c in enumerate(self.cells) if c.id==after), None)
    elif before: idx = next((i for i,c in enumerate(self.cells) if c.id==before), None)
    else: idx = len(self.cells)
    if idx is None: raise KeyError(after or before)
    for i,c in enumerate(cells): self.cells.insert(idx+i, c)

Cells can be moved by id, either relative to another cell or to the end:

In [ ]:
nbo = Notebook.open(minimal_fn)
c0,c1 = nbo[0].id,nbo[1].id
nbo.move(c1, before=c0)
[c.id for c in nbo] == [c1, c0]

True

Use `save` to write to disk:
```py
nbo.save('path.ipynb')
```
If no path is passed, the path used in `open()` will be re-used.

In [ ]:
#| export
@patch
def view(self:Notebook, id, nums=True):
    "Show cell source with optional line numbers"
    lines = self[id].source.splitlines()
    if nums: lines = [f'{i+1:6d} │ {l}' for i,l in enumerate(lines)]
    return '\n'.join(lines)

The `view` method displays a cell's source with optional line numbers:

In [ ]:
print(nbo.view('e2147a69'))

     1 │ # Do some arithmetic
     2 │ 1+1


## Output rendering

In [ ]:
#| export
def preferred_out(data, html1st=True, include_imgs=False):
    preftyps = ('application/javascript', 'text/latex')
    preftyps = (('text/html', 'text/markdown') if html1st else ('text/markdown', 'text/html')) + preftyps
    if include_imgs: preftyps += 'image/jpeg','image/png','image/svg+xml'
    preftyps += ('text/plain',)
    for mt in preftyps:
        if (text := data.get(mt)): return mt,text
    return 'text/plain',''

`preferred_out` selects the best MIME type from an output's `data` dict, preferring HTML by default:

In [ ]:
data = dict(text_plain=['42'], **{'text/html': ['<b>42</b>'], 'text/plain': ['42']})
preferred_out(data), preferred_out(data, html1st=False)

(('text/html', ['<b>42</b>']), ('text/html', ['<b>42</b>']))

In [ ]:
#| export
def apply_controls(text):
    r"Apply \r and \b to text, returning processed result"
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if 0<(rpos := line.rfind('\r'))<len(line)-1: lines[i] = line[rpos+1:]
    text = '\n'.join(lines)
    while (pos := text.find('\b')) >= 0: text = text[:max(0, pos-1)] + text[pos+1:]
    return text

In [ ]:
#| export
def _join(d): return ''.join(d) if isinstance(d, list) else d

In [ ]:
#| export
def mk_stream(name, text):
    "Helper to create an output stream dict"
    return {'output_type': 'stream', 'name': name, 'text': text}

In [ ]:
#| export
def concat_streams(outputs):
    "Concatenate stream outputs by name (stdout/stderr), preserving execute_result at end"
    streams, res, execute_results = {}, [], []
    for out in outputs:
        if out['output_type'] == 'stream':
            name, text = out['name'], _join(out['text'])
            streams[name] = apply_controls(streams.get(name, '') + text)
        elif out['output_type'] in ('error','execute_result'): execute_results.append(out)
        else: res.append(out)
    if 'stdout' in streams: res.append(mk_stream('stdout', streams['stdout']))
    if 'stderr' in streams: res.append(mk_stream('stderr', streams['stderr']))
    res.extend(execute_results)
    return res

`concat_streams` merges consecutive stream outputs by name and moves execute_results to the end, like standard jupyter output rendering:

In [ ]:
outs = [dict(output_type='execute_result', data={'text/plain': ['42']}, metadata={}),
        mk_stream('stdout', 'hello '), mk_stream('stdout', 'world\n'), mk_stream('stderr', 'warn\n')]
outs

[{'output_type': 'execute_result',
  'data': {'text/plain': ['42']},
  'metadata': {}},
 {'output_type': 'stream', 'name': 'stdout', 'text': 'hello '},
 {'output_type': 'stream', 'name': 'stdout', 'text': 'world\n'},
 {'output_type': 'stream', 'name': 'stderr', 'text': 'warn\n'}]

In [ ]:
concat_streams(outs)

[{'output_type': 'stream', 'name': 'stdout', 'text': 'hello world\n'},
 {'output_type': 'stream', 'name': 'stderr', 'text': 'warn\n'},
 {'output_type': 'execute_result',
  'data': {'text/plain': ['42']},
  'metadata': {}}]

In [ ]:
#| export
def _preferred_msg_out(out, **kwargs):
    typ = out['output_type']
    if typ == 'stream': return 'text/plain', _join(out.get('text', ""))
    elif typ == 'error': return 'text/plain', '\n'.join(out.get('traceback', []))
    elif typ in ('execute_result', 'display_data'): return preferred_out(out.get('data', {}), **kwargs)
    return 'text/plain',f'Error: Failed to parse unknown output - {out}'

In [ ]:
#| export
def render_output(out):
    "Convert a single output dict to an HTML string"
    def _fmt(text):
        res = ansi2html(str(text))
        return f'<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">{res}</code></pre>'
    ptyp,d = _preferred_msg_out(out, html1st=True, include_imgs=True)
    d = _join(d)
    if   ptyp=='text/plain': return _fmt(d)
    elif ptyp=='text/html': return d
    elif ptyp=='application/javascript': return f'<script>{d}</script>'
    elif ptyp=='text/markdown': return d
    elif ptyp=='text/latex': return f'<div>{d}</div>'
    elif ptyp=='image/jpeg': return f'<img src="data:image/jpeg;base64,{d}"/>'
    elif ptyp=='image/png':  return f'<img src="data:image/png;base64,{d}"/>'
    elif ptyp=='image/svg+xml': return d
    return ''

In [ ]:
print(render_output(dict(output_type='execute_result', data={'text/plain': ['42']}, metadata={})))

<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">42</code></pre>


In [ ]:
#| export
def render_outputs(outputs):
    "Render a full list of outputs, concatenating streams first."
    if (not isinstance(outputs, (list,tuple))) or (outputs and not isinstance(outputs[0],dict)): return ''
    return '\n'.join(render_output(o) for o in concat_streams(outputs))

In [ ]:
print(render_outputs(outs))

<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">hello world
</code></pre>
<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">warn
</code></pre>
<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">42</code></pre>


## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()